# Maya 2 Native — Indic + English streaming TTS on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Maya2-Native-indic-tts.ipynb)

[`maya-research/maya2-native`](https://huggingface.co/maya-research/maya2-native) is Maya Research's open streaming text-to-speech model (release 1.0, 25–26 October 2026). It is **2.50B parameters** (~5.0 GB of BF16 weights): a [Qwen3-1.7B-Base](https://huggingface.co/Qwen/Qwen3-1.7B-Base) backbone plus a ~555M depth decoder and the [Kyutai Mimi](https://huggingface.co/kyutai/mimi) codec. It writes **24 kHz** mono speech in **80 ms** frames for **11 Indic languages and English**, with **20 built-in synthetic voices** and **zero-shot / cross-lingual cloning** from 3–10 seconds of reference audio.

This is the free-Colab Indic TTS notebook. The [Phonon 31M notebook](phonon-31m-en-streaming-tts.ipynb) is tiny English-only CPU TTS. The [Drama 3 notebook](Drama-3-tts-voice-clone.ipynb) needs a Colab Pro A100.

The product name is **Maya 2 Native**. The published package, class, and config filename are still `maya3`, `Maya3TTS`, and `maya3_config.json`. Use those identifiers.

**Licence:** Apache-2.0 on the weights and the [inference code](https://github.com/MayaResearch/maya2-native). Not gated. Upstream notices still apply: Qwen3-1.7B-Base (Apache-2.0), Mimi (CC-BY-4.0), SpeechBrain ECAPA-TDNN (Apache-2.0).

**Runtime:** *Runtime > Change runtime type > T4 GPU*. A T4 is compute capability 7.5: **no native bfloat16**, no FlashAttention-2, and no sm80-only kernels. The card loads a BF16 backbone, an FP16 depth decoder, Triton attention, and CUDA graphs, and it starts at `max_seats=1` (documented load peak ~9.2 GB plus CUDA context). This notebook keeps `max_seats=1` and a **1024-token KV cache**, runs the backbone in **float16** on T4, uses the engine's eager **gather** attention (not Triton), and turns **backbone CUDA graphs off**. The depth decoder still captures standard `torch.cuda.CUDAGraph`s (that class has no eager path). If a waveform is NaN/Inf, the overflow-prone depth stack is rebuilt in **float32**.

**Links:** [model card](https://huggingface.co/maya-research/maya2-native) · [inference code](https://github.com/MayaResearch/maya2-native) · [engine docs](https://github.com/MayaResearch/maya2-native/blob/main/docs/ENGINE.md) · [voices.json](https://huggingface.co/maya-research/maya2-native/blob/main/voices/voices.json)


## 1. Install `maya3` without replacing Colab's torch

The card's command is `pip install "maya3[server] @ git+https://github.com/MayaResearch/maya2-native" --extra-index-url https://download.pytorch.org/whl/cu128`. That **pins `torch==2.8.0` + `torchaudio==2.8.0`** (cu128) and would reinstall Colab's CUDA torch. This notebook installs the engine with `--no-deps` and only the remaining libraries (`transformers==5.6.2` is the engine pin for Qwen3 + Mimi + the DynamicCache layout).

Do **not** install `flash-attn`. The engine does not use it.


In [ ]:
import subprocess
import sys

import torch

print("python", sys.version.split()[0])
print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
print("cuda available before install:", torch.cuda.is_available())


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


# Card extras [server] (aiohttp) are not used here. Official extra-index install
# pins torch==2.8.0+cu128 and would replace Colab's CUDA torch.
pip_install("maya3 @ git+https://github.com/MayaResearch/maya2-native.git", "--no-deps")
pip_install(
    "transformers==5.6.2",
    "tokenizers>=0.22,<0.23",
    "speechbrain==1.1.2",
    "huggingface_hub>=1.0",
    "safetensors>=0.6",
    "soundfile>=0.12",
    "scipy>=1.13",
    "pyyaml>=6.0",
)

try:
    import torchaudio

    print("torchaudio (unchanged)", torchaudio.__version__)
except ImportError:
    pip_install("torchaudio")
    import torchaudio

    print("torchaudio (just installed, matching Colab torch)", torchaudio.__version__)

print("torch after pip (module still the pre-install wheel until restart)", torch.__version__)


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
>
> Colab has already imported its preinstalled `transformers` (and often `torch`). The `maya3` import and the `transformers==5.6.2` pin are invisible until the process restarts. You do **not** need to rerun the install cell after the restart unless it failed.


## 2. GPU check

Need a CUDA GPU. Print the device name, compute capability, and free VRAM, then pick the T4 dtype path.


In [ ]:
import os
import shutil

import torch
import transformers
from packaging.version import Version

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TRITON_CACHE_DIR", "/tmp/maya3-triton")
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

assert Version(transformers.__version__) >= Version("5.6.2"), (
    f"transformers {transformers.__version__} is loaded; maya3 pins 5.6.2. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the GPU-check cell.")

# T4 (sm_75) has no FlashAttention-2. Keep SDPA on mem-efficient / math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

free_b, total_b = torch.cuda.mem_get_info()
major, minor = torch.cuda.get_device_capability()
props = torch.cuda.get_device_properties(0)

# Card default is BF16 backbone. T4 has no native BF16 — use float16 there.
NATIVE_BF16 = major >= 8
BACKBONE_DTYPE = torch.bfloat16 if NATIVE_BF16 else torch.float16
USE_TRITON = NATIVE_BF16
USE_GRAPHS = NATIVE_BF16
MAX_SEATS = 1
KV_CAPACITY = 1024  # card default is 2048; keep a smaller single-seat KV on 16 GB
DEPTH_DTYPE = "fp16"  # rebuilt as fp32 below if a waveform is non-finite
SAMPLE_RATE = 24_000
MODEL_ID = "maya-research/maya2-native"

print("transformers", transformers.__version__)
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}")
print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
print(
    "path:",
    f"backbone={BACKBONE_DTYPE}".replace("torch.", ""),
    f"| triton={USE_TRITON} | backbone_graphs={USE_GRAPHS}",
    f"| max_seats={MAX_SEATS} | kv={KV_CAPACITY} | depth={DEPTH_DTYPE}",
)
if not NATIVE_BF16:
    print("No native bfloat16 on this GPU; T4 float16 path is on.")
print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")


## 3. Load Maya 2 Native

`Maya3TTS.from_pretrained` is the card API. The engine hardcodes a BF16 backbone, Triton attention, and BF16 prompt embeddings. On compute capability &lt; 8.0 this cell overrides those three sites before load: **float16 backbone**, **gather** attention, **no backbone CUDA graphs**, and prompt / hidden buffers in the same dtype. `max_seats=1` matches the card's starting advice.

The first run downloads the two weight shards (~5.0 GB), `kyutai/mimi`, and `speechbrain/spkrec-ecapa-voxceleb`. Warmup then captures the depth/Mimi graphs and runs two short English phrases.


In [ ]:
import importlib.metadata
from collections import Counter

import numpy as np
import torch
from IPython.display import Audio, display
from maya3 import Maya3TTS

print("maya3", importlib.metadata.version("maya3"))


def apply_t4_runtime_patches(backbone_dtype, *, use_triton, use_graphs):
    # Redirect the three hardcoded BF16 / Triton sites in maya3.engine.
    # The engine always calls load_model_for_eval(..., dtype=bfloat16),
    # builds StaticBackbone(..., attention='triton'), and casts prompts / self.h
    # to bfloat16. StaticBackbone also accepts attention='gather' and
    # use_graphs=False. FastDepth has no eager path, so its CUDA graphs stay on.
    import maya3.engine as engine_mod
    import scripts.s2_serve_controls as serve_controls
    import serve.fast_backbone as fast_backbone
    from tts.model import load_s3

    if getattr(engine_mod.Engine, "_colab_t4_patched", False):
        print("T4 engine patches already applied")
        return

    orig_load = load_s3.load_model_for_eval

    def load_for_eval(*args, **kwargs):
        kwargs["dtype"] = backbone_dtype
        return orig_load(*args, **kwargs)

    load_s3.load_model_for_eval = load_for_eval

    orig_prompt = serve_controls.prompt_from_example

    def prompt_from_example(ex, model, device, dtype, *rest, **kwargs):
        return orig_prompt(ex, model, device, backbone_dtype, *rest, **kwargs)

    serve_controls.prompt_from_example = prompt_from_example

    orig_backbone = fast_backbone.StaticBackbone

    class ColabBackbone(orig_backbone):
        def __init__(self, *args, **kwargs):
            if not use_triton:
                kwargs["attention"] = "gather"
                kwargs["fuse_rope"] = False
                kwargs["fuse_proj"] = False
            kwargs["use_graphs"] = bool(use_graphs)
            super().__init__(*args, **kwargs)

    fast_backbone.StaticBackbone = ColabBackbone

    orig_build = engine_mod.Engine._build_fast

    def build_fast(self):
        orig_build(self)
        if self.h.dtype != backbone_dtype:
            self.h = self.h.to(backbone_dtype)
            self.feedback = torch.zeros_like(self.h)

    engine_mod.Engine._build_fast = build_fast

    orig_to = torch.Tensor.to

    def tensor_to(self, *args, **kwargs):
        if args and args[0] is torch.bfloat16:
            args = (backbone_dtype,) + args[1:]
        if kwargs.get("dtype") is torch.bfloat16:
            kwargs = {**kwargs, "dtype": backbone_dtype}
        return orig_to(self, *args, **kwargs)

    def wrap_bf16_casts(method):
        def wrapped(self, *args, **kwargs):
            torch.Tensor.to = tensor_to
            try:
                return method(self, *args, **kwargs)
            finally:
                torch.Tensor.to = orig_to

        return wrapped

    engine_mod.Engine.prepare_voice = wrap_bf16_casts(engine_mod.Engine.prepare_voice)
    engine_mod.Engine._plan = wrap_bf16_casts(engine_mod.Engine._plan)
    engine_mod.Engine._colab_t4_patched = True


if not NATIVE_BF16:
    apply_t4_runtime_patches(BACKBONE_DTYPE, use_triton=USE_TRITON, use_graphs=USE_GRAPHS)
else:
    print("sm_80+ : card path (BF16 backbone, Triton, CUDA graphs), still max_seats=1")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

tts = Maya3TTS.from_pretrained(
    MODEL_ID,
    max_seats=MAX_SEATS,
    capacity=KV_CAPACITY,
    depth_dtype=DEPTH_DTYPE,
    use_graphs=USE_GRAPHS,
    fuse_rope=USE_TRITON,
    fuse_proj=False,
    fused_norm=False,
    fused_depth=USE_TRITON,
    compile_depth=False,
    free_reference_depth=True,
)

eng = tts.engine
bb = eng.backbone
param_dtypes = Counter(str(p.dtype).replace("torch.", "") for p in eng.model.parameters())
print("model", MODEL_ID)
print("voices", len(tts.voices), ":", ", ".join(tts.voices[:8]), "...")
print("backbone dtype:", bb.dtype, "| attention:", bb.attention, "| graphs:", bb.use_graphs)
print("depth dtype:", eng.depth.dtype, "| fused:", eng.depth.fused)
print("max_seats:", eng.cfg.max_seats, "| kv capacity:", eng.cfg.capacity)
print("parameter dtypes:", dict(param_dtypes))
print(f"load {getattr(eng, 'load_s', float('nan')):.1f}s | warmup {getattr(eng, 'warmup_s', float('nan')):.1f}s")
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB | "
    f"reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

RUNS = []


## 4. Demo 1 — English, built-in voice

`Siddhant` is an adult male voice trained on every language except Assamese, including English. `stream()` yields signed 16-bit PCM at 24 kHz as each 80 ms frame is decoded. `generate()` is the same path concatenated to float32. `normalize=True` is required on the Python API (the HTTP server turns it on by default).


In [ ]:
import time

from serve.fast_depth import FastDepth

EN_TEXT = (
    "Hello from Maya 2 Native. Your order has shipped and will arrive tomorrow."
)
EN_VOICE = "Siddhant"


def rebuild_depth_fp32():
    # Keep the backbone in float16; rebuild the overflow-prone depth stack in float32.

    def _rebuild():
        n = eng.cfg.max_seats
        eng.depth = FastDepth(
            eng.model,
            n,
            dtype=torch.float32,
            batch_buckets=eng.buckets,
            seed=eng.cfg.seed + 1,
            temp=eng.cfg.temp,
            top_k=eng.cfg.top_k,
            compile=False,
            fused=False,
        )
        eng.depth.warmup()
        print("depth decoder rebuilt in float32")

    eng.call(_rebuild)


def speak(text, lang, voice, *, label, seed=20261024):
    info = {}
    t0 = time.perf_counter()
    chunks = list(
        tts.stream(
            text,
            lang=lang,
            voice=voice,
            normalize=True,
            seed=seed,
            info=info,
        )
    )
    elapsed_s = time.perf_counter() - t0
    wav = (
        np.concatenate(chunks).astype(np.float32) / 32767.0
        if chunks
        else np.zeros(0, np.float32)
    )
    if wav.size == 0 or not np.isfinite(wav).all():
        print(f"{label}: non-finite or empty audio; retrying with float32 depth")
        rebuild_depth_fp32()
        info = {}
        t0 = time.perf_counter()
        chunks = list(
            tts.stream(
                text,
                lang=lang,
                voice=voice,
                normalize=True,
                seed=seed,
                info=info,
            )
        )
        elapsed_s = time.perf_counter() - t0
        wav = (
            np.concatenate(chunks).astype(np.float32) / 32767.0
            if chunks
            else np.zeros(0, np.float32)
        )
    if wav.size == 0 or not np.isfinite(wav).all():
        raise RuntimeError(f"{label}: audio is empty or not finite after the fp32 depth retry")
    audio_s = len(wav) / SAMPLE_RATE
    ttfa_s = info.get("ttfa_s")
    rec = {
        "label": label,
        "lang": lang,
        "voice": voice,
        "audio_s": audio_s,
        "wall_s": elapsed_s,
        "ttfa_ms": None if ttfa_s is None else 1000.0 * ttfa_s,
        "frames": info.get("frames"),
        "reason": info.get("reason"),
        "peak_gb": torch.cuda.max_memory_allocated() / 1024**3,
    }
    RUNS.append(rec)
    ttfa = "n/a" if rec["ttfa_ms"] is None else f"{rec['ttfa_ms']:.0f} ms"
    print(
        f"{label}: {lang}/{voice} | {audio_s:.2f}s audio | {elapsed_s:.2f}s wall | "
        f"RTF {elapsed_s / max(audio_s, 1e-6):.2f} | TTFA {ttfa}"
    )
    display(Audio(wav, rate=SAMPLE_RATE))
    return wav


en_wav = speak(EN_TEXT, "en", EN_VOICE, label="English / Siddhant")
print("English samples:", en_wav.shape, "finite:", bool(np.isfinite(en_wav).all()))


## 5. Demo 2 — Hindi and Tamil, built-in voices

`Ritvik` is trained on Hindi. `Kayal` is trained on English and Tamil. Every built-in voice can be asked for any of the 12 languages; languages outside a voice's training list are cross-lingual and may keep an accent. Assamese has no built-in voice with ≥1 hour of training data and is the weakest language on the card — it is not used here.


In [ ]:
HI_TEXT = "नमस्ते, आपका ऑर्डर कल पहुँच जाएगा।"
TA_TEXT = "வணக்கம், எப்படி இருக்கிறீர்கள்?"

hi_wav = speak(HI_TEXT, "hi", "Ritvik", label="Hindi / Ritvik")
ta_wav = speak(TA_TEXT, "ta", "Kayal", label="Tamil / Kayal")
print("Hindi finite:", bool(np.isfinite(hi_wav).all()), "| Tamil finite:", bool(np.isfinite(ta_wav).all()))


## 6. Demo 3 — zero-shot clone from a bundled sample, then cross-lingual

The reference is the card's own [`samples/en_siddhant.wav`](https://huggingface.co/maya-research/maya2-native/blob/main/samples/en_siddhant.wav) (model output, Apache-2.0). It is **not** a recording of a real person. Do not clone an identifiable person's voice without consent.

The card lists the **pre-normalization** input under that clip. `register_voice` wants the exact spoken transcript; spoken dates and abbreviations may differ from this string. That is still the official text shipped with the sample.

`icl_spk` (default) conditions on the reference audio plus its ECAPA embedding. The second generate is **cross-lingual**: English reference, Tamil output.


In [ ]:
from huggingface_hub import hf_hub_download
from maya3.api import load_audio as load_ref

REF_NAME = "samples/en_siddhant.wav"
REF_TEXT = (
    "The EMI for your home loan is due by 12-10-24; please ensure your account "
    "has sufficient funds."
)
CLONE_ID = "siddhant_sample"

ref_path = hf_hub_download(MODEL_ID, REF_NAME)
ref_wav = load_ref(ref_path)
print(f"reference {REF_NAME}: {len(ref_wav) / SAMPLE_RATE:.2f}s, {ref_wav.shape}")
display(Audio(ref_wav, rate=SAMPLE_RATE))

tts.register_voice(CLONE_ID, ref_audio=ref_path, ref_text=REF_TEXT, lang="en")
print("registered", CLONE_ID, "| voices now", len(tts.voices))

clone_en = speak(
    "Your package will arrive tomorrow morning.",
    "en",
    CLONE_ID,
    label="Clone English / bundled Siddhant sample",
)
clone_ta = speak(
    "வணக்கம், எப்படி இருக்கிறீர்கள்?",
    "ta",
    CLONE_ID,
    label="Cross-lingual clone Tamil / bundled Siddhant sample",
)
print(
    "clone finite:",
    bool(np.isfinite(clone_en).all()),
    bool(np.isfinite(clone_ta).all()),
)


## 7. Peak VRAM and timing


In [ ]:
print(f"{'label':<48} {'lang':<4} {'s':>6} {'wall':>6} {'RTF':>5} {'TTFA':>7} {'peak GiB':>9}")
for rec in RUNS:
    rtf = rec["wall_s"] / max(rec["audio_s"], 1e-6)
    ttfa = "—" if rec["ttfa_ms"] is None else f"{rec['ttfa_ms']:.0f}ms"
    print(
        f"{rec['label']:<48} {rec['lang']:<4} {rec['audio_s']:6.2f} "
        f"{rec['wall_s']:6.2f} {rtf:5.2f} {ttfa:>7} {rec['peak_gb']:9.2f}"
    )

free_b, total_b = torch.cuda.mem_get_info()
print(
    f"\nVRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB | "
    f"free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB"
)
print("engine health:", {k: eng.health().get(k) for k in ("ready", "max_seats", "mem_alloc_gb", "mem_reserved_gb")})
print("All demo waveforms were asserted finite at generate time.")


## 8. Caveats

- **T4 dtype.** The card's serving recipe is BF16 backbone + FP16 depth + Triton + CUDA graphs on an H100. A T4 (sm_75) has no native BF16. This notebook loads the backbone in **float16**, uses **gather** attention, and disables **backbone** CUDA graphs and Triton fused kernels. FastDepth / StaticMimi still use `torch.cuda.CUDAGraph` because those classes always capture. On L4 / A100 the cell keeps the card's BF16 + Triton path, still at `max_seats=1`.
- **NaN / Inf audio.** Float16 decode can overflow (the same class of failure we hit with Ming TTS). `speak()` rebuilds only the depth decoder in float32 and retries. The backbone stays float16 so a 16 GB card does not take a full fp32 copy of 2.50B weights.
- **Memory.** Documented single-seat load peak is ~9.2 GB plus CUDA context with a 2048-token KV. This notebook uses `max_seats=1` and `capacity=1024`. Do not raise `max_seats` on a free T4. The first cell downloads ~5 GB of weights plus Mimi and ECAPA.
- **Install / restart.** The official extra-index install pins `torch==2.8.0+cu128` and would replace Colab's torch. This notebook does not. After `transformers==5.6.2` you **must** restart the session once.
- **Names.** Product: Maya 2 Native. Import: `from maya3 import Maya3TTS`. Config file: `maya3_config.json`.
- **Languages.** `as bn en gu hi kn ml mr or pa ta te`. Assamese is the weakest; no built-in voice has an hour of Assamese training audio. One- or two-word prompts drift. URLs, long digit strings, and heavy code-mix need `normalize=True` and still deserve a listen.
- **Cloning.** Similarity is approximate. The bundled sample's listed text is the **pre-normalization** card input. Dedicated clone scores are not in the release. Do not clone a real person without consent. Built-in names are synthetic personas, not real people.
- **No watermark.** The release does not add an audio watermark. Disclose synthetic speech where that matters.
- **Not Phonon, not Drama 3.** Phonon 31M is 64.5 MB English CPU TTS. Drama 3-int4 needs an A100 and GPTQ-Marlin (sm_80+).
- **This file was not executed on a GPU in the pull request that added it.** Numbers above print at runtime on your Colab T4.


## 9. Cleanup

Stop the engine thread and release VRAM when you are done. Re-run the load cell to start again.


In [ ]:
tts.close()
del tts
torch.cuda.empty_cache()
free_b, total_b = torch.cuda.mem_get_info()
print(
    f"after close: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"free / total {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB"
)
